### Developing human heart - subregions of each region (mp_heart_dvlp03)

**Paper**: Lázár E, Mauron R, Andrusivová Ž, Foyer J, He M, Larsson L, Shakari N, Salas SM, Avenel C, Sariyar S, Hansen JN, Vicari M, Czarnewski P, Braun E, Li X, Bergmann O, Sylvén C, Lundberg E, Linnarsson S, Nilsson M, Sundström E, Adameyko I, Lundeberg J. *Spatiotemporal gene expression and cellular dynamics of the developing human heart.* Nat Genet. 2025 Nov;57(11):2756-2771. doi: [10.1038/s41588-025-02352-6](https://doi.org/10.1038/s41588-025-02352-6). PMID: 41162788; PMCID: PMC12597827.

Code: [github.com/rmauron/HDCA_heart_dev](https://github.com/rmauron/HDCA_heart_dev) (Zenodo 15912657) - Data: Mendeley [part 1](https://doi.org/10.17632/fhtb99mdzd.1) (Space Ranger, Cell Ranger, metadata) and [part 2](https://doi.org/10.17632/w65jtfsvpr.1) (Seurat RDS) - [Interactive viewer](https://hdcaheart.serve.scilifelab.se/web/index.html)

For each region of **mp_heart_dvlp01** (**heart01_region_labels.tsv**: Leiden clusters of the cell-type abundances of the 38 sections), the Visium spots of the region are clustered on **gene expression** (own UMAP per region, ComBat by section) to look for **subregions**, which are then **mapped on the tissue**, described and followed **across ages**. It follows **mp_lymphoid03_subregions.ipynb** and **mp_mouse_brain03_subregions.ipynb** (same methods and gene-set definitions, class section **8c. Subregions**); the subregion signatures are scored on the scRNA reference of **mp_heart_dvlp01** (Figure 1).

#### Libs: **src/libs**

- **single_cell_lib**

  - Class **SingleCell**: generic Scanpy, Squidpy, cell2location and LIANA methods

  - **DEFAULT_PARAMS**

- **single_cell_heart_dvlp_lib** (new)

  - Class **SingleCellHeart(SingleCell)**: 
    - developing-heart methods 
      - sections and ages, 
      - the paper's stereoscope proportions and spatial clusters, 
      - gene x age panels, 
      - regions vs the paper, 
      - temporal trends

  - **HEART_PARAMS**, **HEART_LINEAGE** 
    - 31 cell types in 9 lineages
  - **HEART_GENES**
    - 20 heart-development genes
  - **HEART_FILES
    - one file registry per notebook: 'heart01' ... 'heart04'

<br>

| section | content |
|---|---|
| 1 | spots, regions (curated **heart_regions.tsv**) and expression |
| 2 | region signatures (table region); region signature heatmap; region maps |
| 3 | subclustering each region (own UMAP; batch = section) |
| 4 | gene sets of each subregion (partition) |
| 5 | subregion table and diagnostics |
| 6 | each region: UMAP and map of its subregions |
| 7 | map of all subregions (one section per age) |
| 8 | subregions by age |
| 9 | subregion signatures on the scRNA reference |
| 10 | tables |

**Outputs**: Plotly figures (interactive here, saved as HTML in **root_figure**), PNG heatmaps, tables in **root_result**; prefix **heart03_**.

### Critique of the request and how it is handled

1. **Gene sets partition the genes**

  - region signature + subregion signature + not present + other = universe
  - a check asserts it (section 4).

2. **"Not present"**:
  -  detected in >= 30% of the rest of the region, 
  - <= 5% of the subregion, 
  - Wilcoxon BH FDR <= 0.05. 
  - Usually small: a 55-um spot holds ~20 cells of several types.

3. **Circularity.** 
  - Regions come from cell-type abundances; 
  - subregions are clustered on **gene expression**, and abundances only describe them.

4. **38 sections, 16 hearts.** 
  - Subclustering uses HVGs per section and **ComBat by section** (**sub_batch**), 
  - so a subregion is not just one section or heart; 
  - the **single-section** flag (< **section_min_frac** of the spots outside one section) and the per-age table (section 8) show how far each subregion is shared.

5. **Depth.** **depth_ratio**
  - median UMIs of the subregion / region
  - flags subregions made of low- or high-depth spots.

6. **A UMAP cluster is not a subregion until it is mapped**: 
  - every subregion is drawn on the H&E and gets a **spatial coherence ratio** (neighbours in the same subregion / chance).

7. **R18** (20 outlier spots) is not subclustered (**subregion_skip**).

8. **Functional descriptions** 
  - are automatic 
    - signature genes, 
    - enriched cell types, 
    - reference cell types; 
  - anatomical words 
    - are inferences from genes, 
    - not readings of the H&E. 
    - Curated text can be added in **heart_subregion_descriptions.tsv** (**subregion_descriptions**).


## Loading packages

In [ ]:
import os, sys
from pathlib import Path
from importlib.metadata import version
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import gc

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import rcParams

rcParams['pdf.fonttype'] = 42          # editable text in PDFs
%config InlineBackend.print_figure_kwargs = {'bbox_inches': 'tight', 'dpi': 90}   # inline display only; files are saved at 300 dpi

import plotly.io as pio
pio.renderers.default = "notebook_connected"

ROOT0 = Path('/home/flavio/uv/cell_cell_communication')
ROOT_SRC = ROOT0 / 'src'
if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

from libs.single_cell_heart_dvlp_lib import SingleCellHeart, HEART_PARAMS, HEART_LINEAGE, HEART_GENES, HEART_FILES
from libs.single_cell_lib import cmap_byr, cmap_lbwr
from libs.matplotlib_figure_style import apply_figure_style
from libs.Basic import create_dir, pdreadcsv

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 250)
pd.set_option('display.max_rows', 200)
print({p: version(p) for p in ['scanpy', 'anndata', 'plotly']})

#3# Parameters

- **HEART_PARAMS** / **DEFAULT_PARAMS** 
  - subclustering: **sub_**;

  - gene sets and flags: 
    - **region_sig_**, **sub_sig_**, **notpres_**, **border_**, ...; 

    - files **HEART_FILES['inputs']** + **['heart03']**. 

- Every method that writes a table has **force** / **verbose**: 

  - with **force=False** an existing table is read instead of recomputed; set *verbose=Truese** after changing parameters or inputs.

In [ ]:
force=False
verbose=True

apply_figure_style()
sch = SingleCellHeart(ROOT0, analysis='heart03')
print(sch)
sch.calc_df_files()

### 1. Spots, regions and expression

Regions of mp_heart_dvlp01 -> **obs['region_label']** (R00 ...); **prepare_expression**: gene symbols as **var_names**, raw counts in **layers['counts']**, X = log1p(CP10k). Abundances: the source the regions were made with.

In [ ]:
df_regions = sch.read_tsv('region_labels', verbose=verbose)

print(df_regions.shape)
df_regions.head(3)

In [ ]:
source = df_regions['abundance_source'].iloc[0]
adata_vis = sch.read_h5ad_slots('c2l_map', skip=('layers',), verbose=verbose) if source == 'c2l' else \
            sch.read_h5ad('visium', set_self=False, verbose=verbose)
adata_vis = sch.remove_mt_genes(adata_vis)
names_auto = sch.add_region_labels_from_table(df_regions, adata_vis)

In [ ]:
df_abund = sch.get_abundance(adata_vis, source=source)
df_region_info = sch.read_region_info(source=source)
print(df_region_info.shape)
df_region_info.head(6).T

### Extra table Heart Lineage (Create by Claude-Opus 5.5)

Dicitionary of heart cell types

In [ ]:
fname = 'heart_lineage.tsv'

root_table = ROOT0 / 'tables'
dfhl = pdreadcsv(fname, root_table, verbose=True)
dfhl.head(5).T

In [ ]:
names = dict(zip(df_region_info.region_id, df_region_info.region_name)) if df_region_info is not None else names_auto

adata_expr = sch.prepare_expression(adata_vis)
sections_rep = sch.representative_sections()
sections_show = [s for s in sections_rep if sch.section_title(s, short=True).split(' | ')[0] in ('w6', 'w8', 'w10', 'w12')]
print(adata_expr)
print("abundance source:", source, "| sections shown in the per-region figures:", sections_show)
adata_expr.obs['region_label'].value_counts().sort_index().to_frame('n_spots').T

In [ ]:
adata_expr.obs.head(3).T

### sequencing depth and cell density (abundance per spot: cells for cell2location, 1 for stereoscope proportions)

In [ ]:
df_depth = sch.calc_df_region_depth(adata_expr, df_abund)
df_depth.T

### 2. Region signatures (table region)

Each region vs all other spots (Wilcoxon on log1p CP10k):

  - signature = logFC >= 1, FDR <= 0.05, detected in >= 25% of the region's spots. 
  - Enriched cell types: mean abundance in the region / in all other spots.
  - Region names and compartments: **heart_regions.tsv**.


In [ ]:
'''
df = sch.calc_df_rank_genes(adata_expr, groupby="region_label")
df = df.rename(columns={"group": "region_id", "names": "gene"})
df
'''

### Rank region - Region Signatures

In [ ]:
force=False
verbose=False

# "rest" (default): all other cells/spots pooled ("one vs rest").
# each Rxx verssus all others Rx1, Rx2, Rxn
# "wilcoxon" test: rank-based (Mann–Whitney U); robust to outliers and the usual choice for single-cell data.
df_rank_region, region_sig = sch.calc_df_region_signature(adata_expr, force=force, verbose=verbose)

print(np.array([x for x in region_sig['R00']]))

print(df_rank_region.shape)
df_rank_region.head(10)

### Region Table

In [ ]:
df_region_table = sch.calc_df_region_table(adata_expr, df_rank_region, region_sig, df_abund, df_region_info,
                                           force=force, verbose=verbose)
print(np.array(df_region_table.columns))

cols = ['region_id', 'region_name', 'compartment', 'n_spots', 'median_UMIs',  'median_cells_per_spot', 
        'n_signature_genes', 'signature_genes', 'enriched_cell_types', 'functional_description']
df_region_table[cols].head(4).T

In [ ]:
force=False
verbose=True

fig = sch.plotly_region_signature_heatmap(adata_expr, df_rank_region, n_per_region=3, fontsize_title=22, fontsize_xticks=14,
                                          fontsize_yticks=13, fontsize_colorbar=14, fontsize_hover=13, height=1100, width=1300,
                                          force=force, verbose=verbose)
if fig: 
    fig.show()

**Region map** of the w8 section (all regions on the H&E; click a legend entry to hide a region, double-click to isolate it). The other representative sections are written as HTML files.

In [ ]:
sections_rep

In [ ]:
sch.params['library_main']

In [ ]:
force=False
verbose=True

lib = sch.params['library_main']

fig = sch.plotly_region_map(adata_expr, adata_vis, lib, df_region_table=df_region_table, fontsize_title=20, fontsize_legend=12,
                            fontsize_hover=13, reset_button=True, force=force, verbose=verbose)

if fig:
    fig.show()

### 3. Subclustering each region (own UMAP)

**subcluster_adaptive** per region: genes detected in >= 2% of the region's spots, 2,000 HVGs (seurat_v3, per section), ComBat by section, PCA (20), kNN (15), UMAP, Leiden over **sub_resolutions**. The resolution kept is the highest at which every subregion has >= **sub_min_size** spots and there are <= **sub_max_k** subregions.

In [ ]:
force=False
verbose=True

subs, df_grid = sch.run_subregions(adata_expr, force=force, verbose=verbose)
cols = ['region_id', 'resolution', 'k', 'min_size']
df_grid[df_grid.chosen][cols].head(8)

In [ ]:
subs

### 4. Subregions

#### Gene sets of each subregion (partition)

Per subregion (Wilcoxon, subregion vs rest of its region), universe = genes detected in >= 1% of the region's spots:

| set | definition |
|---|---|
| region_signature | in the region signature (section 2) |
| subregion_signature | logFC >= 0.75, FDR <= 0.05, detected in >= 20% of the subregion; not in the region signature |
| not_present | detected in >= 30% of the rest of the region, <= 5% of the subregion, FDR <= 0.05 |
| other | the rest of the universe |

In [ ]:
force=False
verbose=True

df_gene_sets = sch.calc_df_subregion_gene_sets(adata_expr, region_sig, force=force, verbose=verbose)
df_set_counts = df_gene_sets.groupby(['sub_region_id', 'gene_set'], sort=False).size().unstack(fill_value=0)
df_set_counts['total'] = df_set_counts.sum(1)

n_universe = df_gene_sets.groupby('sub_region_id', sort=False).size()
assert (df_set_counts['total'] == n_universe.loc[df_set_counts.index]).all()

print(f"{len(df_set_counts)} subregions; the 4 sets partition every universe")

df_set_counts.head(10)

### 5. Subregion table

Diagnostics:

- **coherence_ratio** (spatial contiguity; 1 = random), 
- **depth_ratio**, 
- **border_region** / **border_frac** / **border_rel**, **status** (supported / weak + flags such as single-section). 

- Functional descriptions: 
  - **heart_subregion_descriptions.tsv** when present, otherwise automatic.

In [ ]:
force=False
verbose=True

path_desc = sch.file_path('subregion_descriptions')
df_descriptions = sch.read_tsv('subregion_descriptions', verbose=verbose) if path_desc.exists() else None
df_sub_table = sch.calc_df_subregion_table(adata_expr, df_gene_sets, df_abund, df_descriptions=df_descriptions,
                                           force=force, verbose=verbose)
print(df_sub_table.status.str.split(';').str[0].value_counts().to_dict())
cols = ['sub_region_id', 'n_spots', 'coherence_ratio', 'depth_ratio', 'border_region', 'border_frac', 'border_rel',
        'n_subregion_signature', 'n_not_present', 'status']
df_sub_table[cols].head(12)

In [ ]:
force=False
verbose=True

fig = sch.plotly_subregion_diagnostics(df_sub_table, fontsize_title=22, fontsize_axis=18, fontsize_ticks=16, fontsize_legend=12,
                                       fontsize_legend_title=16, fontsize_hover=13, grid_color="#555555", height=800, width=1800,
                                       force=force, verbose=verbose)
if fig:fig.show()

### 6. Each region: UMAP and map of its subregions

Per region: UMAP coloured by subregion, UMAP coloured by section, and the subregions on the H&E of four sections (w6, w8, w10, w12; a region spans up to 38 sections, so all of them in one figure would be unreadable). The UMAPs show every spot of the region. Click a legend entry to hide / isolate a subregion in all panels.

In [ ]:
subs

In [ ]:
force=False
verbose=True

adata_show = adata_vis[adata_vis.obs['section'].isin(sections_show)].copy()
adata_show.uns['spatial'] = {k: v for k, v in adata_vis.uns['spatial'].items() if k in sections_show}

cnt = 0
for region, adata_sub in subs.items():
    fig = sch.plotly_region_subregions(region, adata_sub, adata_show, 
                                       df_sub_table=df_sub_table, df_region_table=df_region_table,
                                       ncols=2, nrows=3,
                                       fontsize_title=20, fontsize_subplot_title=16, fontsize_axis=14, fontsize_legend=12,
                                       fontsize_colorbar=12, fontsize_hover=13, force=force, verbose=verbose)
    if fig: fig.show()

    cnt += 1
    if cnt == 2: break

### 7. Map of all subregions

Every subregion in a shade of its region's colour (legend grouped by region). Hover: status, signature genes, absent genes, enriched cell types. One section per age; the w8 section here, the others as HTML files.

In [ ]:
force=False
verbose=True

# for lib in sections_rep:
lib = sch.params['library_main']
fig = sch.plotly_subregion_map(adata_expr, adata_vis, lib, df_sub_table=df_sub_table, fontsize_title=20, fontsize_legend=11,
                                fontsize_legend_title=14, fontsize_hover=13, legend_ncols=3, width=1800, height=900,
                                force=force, verbose=verbose)
if fig:
    fig.show()

### 8. Subregions by age

Share of each subregion among the spots of each section, mean over the sections of an age (as the regions in mp_heart_dvlp02). Rows scaled to their maximum; the text is the share in %. A subregion present at one age only (or one heart) is a candidate stage-specific structure - or a section effect; check the **single-section** flag.

In [ ]:
force=False
verbose=True

adata_vis.obs['sub_region_id'] = adata_expr.obs['sub_region_id'].reindex(adata_vis.obs_names).astype(str)

df_sub_age = sch.calc_df_region_age(adata_vis[adata_vis.obs['sub_region_id'] != 'nan'], key='sub_region_id',
                                    fname=sch.fname('tsv_subregion_age'), force=force, verbose=verbose)

fig = sch.plot_group_age_heatmap(df_sub_age, group_col='region', figsize=(10, 22), title='Subregion share by age (row-scaled)',
                                 fname=sch.fname('fig_subregion_age'), force=force, verbose=verbose)
plt.show()

### 9. Subregion signatures on the scRNA reference

Which reference cell types express each subregion's signature? **sch.calc_df_subregion_ref_score**: for every **supported** subregion, the top 25 subregion-signature genes found in the reference (>= 3 genes); cell score = mean per-gene z-score of log1p(CP10k); mean per cell type. Rare types with very specific genes reach high scores; compare cell types **within a row**.

In [ ]:
force=False
verbose=True

adata_ref = sch.filter_reference(sch.read_h5ad('reference', set_self=False))
adata_ref_sym = adata_ref.copy()
adata_ref_sym.var_names = adata_ref_sym.var['symbol'].astype(str).values
adata_ref_sym.var_names_make_unique()

df_ref_score, df_cell_score, genes_used = sch.calc_df_subregion_ref_score(adata_ref_sym, df_gene_sets, df_sub_table,
                                                                          force=force, verbose=verbose)
ref_top = sch.calc_df_ref_top(df_ref_score, n=3)
print(df_ref_score.shape, "(subregions x cell types)")

ref_top.to_frame()

In [ ]:
force=False
verbose=True

fig = sch.plot_subregion_ref_heatmap(df_ref_score, figsize=(16, max(6, 0.22 * len(df_ref_score))), fname=sch.fname('fig_ref_heatmap'),
                                     force=force, verbose=verbose)
plt.show()

### 10. Tables

**Table region**

In [ ]:
force=False
verbose=True

df_region_table = sch.update_region_table_subregions(df_region_table, adata_expr, force=force, verbose=verbose)
df_region_table

**Table subregion** (with the reference cell types of section 9; '' = not scored)

In [ ]:
force=False
verbose=True

df_sub_table = sch.add_ref_cell_types(df_sub_table, df_ref_score, n=3, force=force, verbose=verbose)

cols = ['region_id', 'sub_region_id', 'status', 'subregion_signature_genes', 'not_present_genes',
        'enriched_cell_types', 'ref_cell_types', 'functional_description']

df_sub_table[cols]

### Notes for interpretation

- **supported** = spatially coherent (ratio >= 1.5) with >= 3 own signature genes; **weak** subregions are not distinct from the rest of their region.

- **Laterality and layers.** Ventricular regions (compact, trabecular) are compositionally alike in LV and RV, so their subregions often split **left vs right** (*HAND1*, *TBX5* vs *HAND2*, *IRX* genes) or **compact vs trabecular** (*HEY2* vs *NPPA*, *IRX3*): the expression differences the abundance clustering could not see. Compare with the paper's clusters (LV_C, RV_C, LV_T, RV_T; mp_heart_dvlp02).
- **Blood subregions.** Subregions whose signature is haemoglobin genes (*HBA1/2*, *HBG1/2*, *HBB*, *ALAS2*, *AHSP*, *SLC4A1*; e.g. R01_2, R03_5, R07_5) are blood in the chamber and vessel lumens (the paper's clusters **B_A** / **B_V**, blood in atria / ventricles). The authors removed the red-blood-cell clusters (HL_excl_*) from the reference, so these spots have no matching cell type: their abundances are spread over the remaining types, and their reference scores (section 9) are low and unspecific. They can be excluded from the LR analysis of mp_heart_dvlp04 by their status or signature.
- **Other recurring structures.** Macrophage (*C1QA*, *CSF1R*, *LYVE1*), platelet (*PF4*, *PPBP*, *ITGA2B*), vascular (*CD93*, *ESAM*, *PDGFB*) and fibroblast / valve (*POSTN*, *PRRX1*, *COL1A1*) subregions appear inside several myocardial regions: tissue elements that the abundance regions merged into the myocardium.
- **Developmental stage.** Section 8 separates subregions shared by all ages from stage-specific ones; with 1-3 hearts per age, a stage-specific subregion should be checked heart by heart.
- **Parameters.** Subregion IDs and gene sets depend on the parameters and on the regions (abundance source); after a change, rerun with **force=False** and recheck the descriptions.